In [1]:
pip install pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.8/742.8 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 11.5 MB/s eta 0:00:00


In [30]:
from pinecone import Pinecone, ServerlessSpec
import csv
import os
import cv2
import urllib.request
import numpy as np
import requests
import time
import joblib
import pandas as pd

In [3]:
from google.colab import userdata
PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')

In [79]:
def get_db(name, embed_dim, metric):
    pc = Pinecone(api_key=PINECONE_API_KEY)

    # Create index if it doesn't exist
    existing_indexes = pc.list_indexes().names()
    if name not in existing_indexes:
        spec = ServerlessSpec(
            cloud="aws",
            region="us-east-1",
        )
        pc.create_index(
            name=name,
            dimension=embed_dim,
            metric=metric,  # or "euclidean"
            spec=spec,
        )
        print(f"Created Pinecone index: {name}")
    else:
        print(f"Pinecone index ready: {name}")

    # Return the index object
    index = pc.Index(name=name)
    return index

In [8]:
def download_imgs(csv_path: str):
    """
    Reads products CSV, generates embeddings in batches,
    and upserts them into Pinecone.
    """
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV not found: {csv_path}")

    # Read CSV and download image as jpg from url column
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            product_id = row.get("reference")
            name = row.get("name", "")
            brand = row.get("brand", "")
            category = row.get("category_hint", "")
            color = row.get("color", "")
            image_url = row.get("image_url", "")
            session = requests.Session()
            session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'image/avif,image/webp,image/apng,image/svg+xml,image/*,*/*;q=0.8',
    'Referer': 'https://www.zara.net/'
})
            for attempt in range(3):
              try:
                response = session.get(image_url, timeout=15)
                if response.status_code == 200:
                    with open(f'/content/dataset/clothes/{product_id}.jpg', 'wb') as f:
                        f.write(response.content)
                        break
                elif response.status_code == 403:
                # If we hit a 403, wait longer
                    time.sleep(2 ** attempt)
                    if attempt == 2:
                      print(f'Hit 403 after 3 attempts, skipping {product_id}.')
              except:
                pass

In [6]:
os.makedirs('/content/dataset/clothes/', exist_ok=True)

In [9]:
download_imgs('/content/zara_combined.csv')

In [11]:
!ls /content/dataset/clothes | wc -l

3551


In [12]:
!zip -r zara_imgs.zip /content/dataset/clothes

  adding: content/dataset/clothes/ (stored 0%)
  adding: content/dataset/clothes/496091357.jpg (deflated 0%)
  adding: content/dataset/clothes/465580420.jpg (deflated 2%)
  adding: content/dataset/clothes/497562252.jpg (deflated 0%)
  adding: content/dataset/clothes/467644328.jpg (deflated 0%)
  adding: content/dataset/clothes/479094967.jpg (deflated 0%)
  adding: content/dataset/clothes/468903719.jpg (deflated 1%)
  adding: content/dataset/clothes/481503261.jpg (deflated 0%)
  adding: content/dataset/clothes/473233325.jpg (deflated 0%)
  adding: content/dataset/clothes/468048976.jpg (deflated 5%)
  adding: content/dataset/clothes/460046226.jpg (deflated 0%)
  adding: content/dataset/clothes/496091365.jpg (deflated 0%)
  adding: content/dataset/clothes/458806357.jpg (deflated 0%)
  adding: content/dataset/clothes/484898416.jpg (deflated 0%)
  adding: content/dataset/clothes/452735648.jpg (deflated 0%)
  adding: content/dataset/clothes/486910316.jpg (deflated 8%)
  adding: content/datas

In [13]:
import os
from google.colab import drive

drive.mount('/content/drive')

save_path = '/content/drive/MyDrive/agents_project2/zara'

if not os.path.exists(save_path):
    os.makedirs(save_path)
    print(f"Created directory: {save_path}")

!cp zara_imgs.zip /content/drive/MyDrive/agents_project2/zara

Mounted at /content/drive
Created directory: /content/drive/MyDrive/agents_project2/zara


Color Clean Up

In [55]:
df = pd.read_csv("zara_combined.csv")

In [59]:
list(df['color'].unique())

['Black',
 'Khaki',
 'Brown',
 'Dark indigo',
 'Chocolate',
 'brown vigore',
 'Black gold',
 'Golden',
 'taupe brown',
 'Ecru / Black',
 'Burgundy',
 'Ocher',
 'Mid khaki',
 'Dark brown',
 'Navy blue',
 'Green',
 'Light khaki',
 'Brown / Ecru',
 'Beige',
 'Mink brown',
 'Ecru',
 'Dusty pink',
 'Taupe gray',
 'Blue',
 'Light mink',
 'Eggplant',
 'Mid-blue',
 'whiskey',
 'Gray',
 'Wine',
 'Chocolate brown',
 'Black / Brown',
 'Light brown',
 'Maroon',
 'Dark khaki',
 'Sand / Marl',
 'Dark maroon',
 'Olive green',
 'Sand',
 'Anthracite Gray',
 'Black / Ecru',
 'Ecru / Khaki',
 'Indigo',
 'Light beige',
 'Mustard',
 'Charcoal',
 'Gray / Tan',
 'Multicolored',
 'Dark camel',
 'Caramel',
 'Dark burgundy',
 'Tobacco Brown',
 'Light blue',
 'Olive Green',
 'Black / Green',
 'Light gray',
 'Mink',
 'Dark green',
 'Bottle green',
 'Toffee',
 'Mid-camel',
 'Butter',
 'Light tan',
 'Black / White',
 'Brown / Taupe',
 'Plum',
 'Champagne',
 'Brown marl',
 'Dark gray',
 'Gray marl',
 'Burnt orange',

In [69]:
import webcolors

def fashion_to_rgb(name):
    if not isinstance(name, str) or name.isdigit():
        return (128, 128, 128) # Gray for invalid/numeric data

    name = name.lower().strip()

    # 1. Direct Overrides for "Fashion Only" Terms
    fashion_dictionary = {
        'vigore': (93, 64, 55),      # Heathered Brown
        'ecru': (250, 249, 246),    # Off-white/Cream
        'tobacco': (110, 66, 30),   # Warm Earthy Brown
        'whiskey': (167, 85, 2),    # Amber Brown
        'anthracite': (56, 56, 56), # Dark Charcoal
        'mink': (136, 119, 105),    # Muted Taupe
        'taupe': (139, 133, 137),   # Gray-Brown
        'burgundy': (128, 0, 32),   # Deep Red
        'khaki': (195, 176, 145),   # Tan-Khaki
        'bottle green': (0, 106, 78)
    }

    # 2. Check for multi-colors (e.g., 'Ecru / Black')
    # Strategy: Take the first dominant color mentioned
    if '/' in name:
        name = name.split('/')[0].strip()

    # 3. Keyword Match (Strip "Dark", "Light", "Mid")
    for fashion_key, rgb in fashion_dictionary.items():
        if fashion_key in name:
            return rgb

    # 4. Standard Color Match (Using webcolors)
    standard_colors = ['red', 'blue', 'green', 'yellow', 'black', 'white', 'pink', 'purple', 'gray', 'orange', 'brown', 'indigo', 'maroon', 'silver', 'tan']
    for base in standard_colors:
        if base in name:
            try:
                return webcolors.name_to_rgb(base)
            except:
                return (128, 128, 128)

    return (128, 128, 128) # Final Fallback

Preprocess & Add to Pinecone

In [5]:
import os
from google.colab import drive

drive.mount('/content/drive')

read_path = '/content/drive/MyDrive/agents_project2/zara'

Mounted at /content/drive


In [6]:
import zipfile
import os

zip_file_name = 'zara_imgs.zip'
zip_file_path = os.path.join(read_path, zip_file_name)
extract_dir = os.path.join('/content', 'zara_imgs')

# Create the extraction directory if it doesn't exist
os.makedirs(extract_dir, exist_ok=True)

# Unzip the file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"Successfully unzipped '{zip_file_name}' to '{extract_dir}'")

Successfully unzipped 'zara_imgs.zip' to '/content/zara_imgs'


In [8]:
img_dir = '/content/zara_imgs/content/dataset/clothes'

Pinecone index ready: product-non-dl-features


In [7]:
# copied from nb1, can put in utils folder
def clahe_grayscale(img):
  clahe = cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8, 8))
  img = clahe.apply(img)
  return img

def add_padding(cropped, height, width):
  diff = np.abs(height - width)
  p1 = diff // 2
  p2 = diff - p1
  if height > width:
    padded = cv2.copyMakeBorder(cropped, 0, 0, p1, p2, cv2.BORDER_CONSTANT, value=[0, 0, 0])
  else:
    padded = cv2.copyMakeBorder(cropped, p1, p2, 0, 0, cv2.BORDER_CONSTANT, value=[0, 0, 0])
  return padded

In [70]:
def color_and_category(img_id):
  item = df.loc[df['reference'] == int(img_id)]
  category = item['category'].values[0]
  color = item['color'].values[0]
  cleaned_color_rgb = fashion_to_rgb(color)
  return category, cleaned_color_rgb


In [85]:
import os
import cv2
import numpy as np
from tqdm import tqdm

def upsert_images_batched(f_index, c_index, image_dir, pca, scaler, batch_size=100):
    shape_batch = []
    color_batch = []

    hog = cv2.HOGDescriptor(_winSize=(128, 128), _blockSize=(16, 16),
                            _blockStride=(8, 8), _cellSize=(8, 8), _nbins=9)

    file_list = [f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    for i, img_name in enumerate(tqdm(file_list, desc="Processing Batches")):
        img_id = img_name.split('.')[0]
        img_path = os.path.join(image_dir, img_name)

        # --- Preprocessing ---
        img = cv2.imread(img_path)
        if img is None: continue

        height, width = img.shape[:2]
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        normalized = clahe_grayscale(gray)
        padded = add_padding(normalized, height, width)
        final = cv2.resize(padded, (128, 128))

        hog_features = hog.compute(final).reshape(1, -1)
        post_pca = pca.transform(hog_features)
        post_scaling = scaler.transform(post_pca)
        final_hog_vec = post_scaling.flatten().tolist()

        # Skip any all 0 vectors for now
        if np.all(final_hog_vec==0):
          print(f'Skipping a feature vectors ith all 0s for {img_id}')
          continue

        category, cleaned_color_rgb = color_and_category(img_id)
        metadata = {"item_type": str(category)}
        epsilon = 1e-6
        safe_color_vec = [float(x) if x > 0 else epsilon for x in cleaned_color_rgb]

        # --- Add to Buffers ---
        shape_batch.append({
            "id": img_id,
            "values": final_hog_vec,
            "metadata": metadata
        })

        color_batch.append({
            "id": img_id,
            "values": safe_color_vec,
            "metadata": metadata
        })

        # --- Upsert ---
        if len(shape_batch) >= batch_size:
            f_index.upsert(vectors=shape_batch)
            c_index.upsert(vectors=color_batch)
            # Clear buffers
            shape_batch = []
            color_batch = []

    # --- Final Cleanup ---
    if shape_batch:
        f_index.upsert(vectors=shape_batch)
        c_index.upsert(vectors=color_batch)

    print(f"\nSuccessfully indexed {len(file_list)} items.")

In [86]:
pca = joblib.load('/content/pca_model.joblib')
scaler = joblib.load('/content/scaler_model.joblib')

features_index = get_db("product-non-dl-features", 1655, metric='cosine')
color_index = get_db("product-non-dl-colors", 3, metric='euclidean')

Created Pinecone index: product-non-dl-features
Pinecone index ready: product-non-dl-colors


In [87]:
upsert_images_batched(features_index, color_index, img_dir, pca, scaler)

Processing Batches: 100%|██████████| 3551/3551 [10:43<00:00,  5.52it/s]



Successfully indexed 3551 items.
